In [1]:
import json
from transformers import AutoTokenizer
from math_verify import parse, verify, LatexExtractionConfig, ExprExtractionConfig

def compile_predictions(predictions_file, model_name):
    extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    results = []
    total_tokens = 0
    with open(predictions_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            gold = parse(f"${data['answer']}$", extraction_config=extraction_target)
            
            # Get model generation output (usually first item)
            llm_output = data['model_generation'][0] if isinstance(data['model_generation'], list) else data['model_generation']
            answer = parse(llm_output, extraction_config=extraction_target)
            total_tokens += len(tokenizer.encode(llm_output))
            result = verify(gold, answer)
            results.append(result)
    
    accuracy = sum(results) / len(results) if results else 0
    avg_tokens = total_tokens / len(results) if results else 0
    return accuracy, avg_tokens

from math import comb
def pass_at_k(n, c, k):
    """
    n: 总样本数
    c: 通过的样本数
    k: 评估的候选数
    """
    if n < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

def compile_predictions_BoN_passK(predictions_file):
    extraction_target = (ExprExtractionConfig(), LatexExtractionConfig())
    results = []
    with open(predictions_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            gold = parse(f"${data['answer']}$", extraction_config=extraction_target)
            
            o_results = []
            # Use all generations to verify
            for llm_output in data['model_generation']:
                answer = parse(llm_output, extraction_config=extraction_target)
                result = verify(gold, answer)
                o_results.append(result)
            results.append(o_results)
    
    # 计算BoN - 检查每个问题是否至少有一个正确
    BoN = sum(1 for o_result in results if any(o_result)) / len(results) if results else 0
    
    #  计算Pass@K - 统计所有通过的样本
    total_samples = sum(len(o_result) for o_result in results)
    correct_samples = sum(sum(o_result) for o_result in results)
    passK = pass_at_k(total_samples, correct_samples, 1)
    
    return BoN, passK


/media/volume/llm/miniconda3/envs/easysteer/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import matplotlib.pyplot as plt

def draw_comparison_fig(baseline_x, baseline_y, seal_x, seal_y, x_label, y_label, title):
    plt.figure(figsize=(10, 5))
    plt.plot(baseline_x, baseline_y, marker='o', linestyle='-', color='b', label='Baseline')
    plt.plot(seal_x, seal_y, marker='o', linestyle='-', color='r', label='SEAL')
    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    plt.ylim(bottom=0, top=1)
    plt.legend()
    plt.show()

## Example

In [ ]:
import os
paths = ["/media/volume/llm/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_test/1/0/10000/predictions.jsonl",
         "/media/volume/llm/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/MATH500/1/0/10000/predictions.jsonl",
         "/media/volume/llm/llm_steering_reasoning/results/deepseek-seal/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_test/1/0/10000/1/predictions.jsonl",
         "/media/volume/llm/llm_steering_reasoning/results/deepseek-seal/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/MATH500/1/0/10000/1/predictions.jsonl"]
for path in paths:
    metric_path = os.path.join(os.path.dirname(path), "metrics.json")
    accuracy, avg_tokens = compile_predictions(path, "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
    print("accuracy: ", accuracy, "avg_tokens: ", avg_tokens)
    # write to metric_path
    with open(metric_path, 'w') as f:
        json.dump({"accuracy": accuracy, "avg_tokens": avg_tokens}, f)


accuracy:  0.7816527672479151 avg_tokens:  2106.838514025777
accuracy:  0.712 avg_tokens:  4824.974
accuracy:  0.8043972706595905 avg_tokens:  1217.3017437452615
accuracy:  0.798 avg_tokens:  3310.358
